# Seaborn 终极复习手册

> 涵盖：relplot → 样式 → 配色 → melt/pivot → 热力图(clustermap) → 柱状图 → 箱线/提琴 → 分布(hist/kde/rug/ecdf) → jointplot → pairplot → FacetGrid → regplot/lmplot → catplot → 树形图 → 速查表

> **核心**：Seaborn = Matplotlib 高级封装。**hue/style/size/col/row** 四个参数是灵魂。
> **Figure-level** (relplot/displot/catplot/lmplot/jointplot/pairplot) 自动创建 Figure 并支持分面；**Axes-level** (scatterplot/boxplot/regplot/...) 可嵌入 plt.subplots。

## 0. 环境准备

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f'Seaborn {sns.__version__}')

---
## 1. Seaborn vs Matplotlib

Seaborn 的核心优势：一行 `hue=` 完成 Matplotlib 需要多行循环/分组才能做的事。

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv', index_col=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matplotlib — 需要手动分组
for d in salary['District'].unique():
    s = salary[salary['District'] == d]
    axes[0].scatter(s['Salary'], s['Age'], label=d, alpha=0.6)
axes[0].set_title('Matplotlib — 手动for循环分组'); axes[0].legend()

# Seaborn — hue 一行搞定
sns.scatterplot(x='Salary', y='Age', hue='District', data=salary, ax=axes[1])
axes[1].set_title('Seaborn — hue一行')

plt.tight_layout(); plt.show()

---
## 2. relplot — 关系图（Seaborn 最核心函数）

**Figure-level**（自动创建 Figure）。`kind='scatter'` 默认，`kind='line'` 折线。

| 参数 | 作用 |
|------|------|
| `hue` | 颜色分组 |
| `style` | 点形/线形分组 |
| `size` | 点大小映射 |
| `col` / `row` | 分面（分子图） |
| `col_wrap` | 每行几个子图 |
| `hue_order` / `col_order` | 控制分组顺序 |

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv', index_col=0)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

sns.scatterplot(x='Salary', y='Age', data=salary, ax=axes[0,0])
axes[0,0].set_title('1) 基础散点')

sns.scatterplot(x='Salary', y='Age', hue='District', data=salary, ax=axes[0,1])
axes[0,1].set_title('2) hue=颜色')

sns.scatterplot(x='Salary', y='Age', style='Gender', data=salary, ax=axes[0,2])
axes[0,2].set_title('3) style=形状')

sns.scatterplot(x='Salary', y='Age', hue='Education', style='Gender', data=salary, ax=axes[1,0])
axes[1,0].set_title('4) hue+style 组合')

sns.scatterplot(x='Salary', y='Age', size='Salary', data=salary, ax=axes[1,1])
axes[1,1].set_title('5) size=大小映射')

sns.lineplot(x='Age', y='Salary', hue='Education', data=salary, ax=axes[1,2])
axes[1,2].set_title('6) lineplot 折线')

plt.tight_layout(); plt.show()

### relplot 分面：col / row

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv', index_col=0)

# col= 按列分面, col_wrap= 每行几个
sns.relplot(x='Salary', y='Age', col='District', data=salary,
            col_wrap=2, height=3.5)
plt.suptitle('col=District 分面', y=1.02)
plt.show()

In [ ]:
# 终极组合：col + row + hue + style
sns.relplot(x='Salary', y='Age', col='Gender', row='District',
            hue='Education', style='Education',
            data=salary, height=3)
plt.show()

### 排序：用 score map 让子图按自定义顺序排列

In [ ]:
score_map = {'Only language': 5, 'Very well': 4, 'Well': 3, 'Not well': 2, 'Not at all': 1}
salary2 = salary.copy()
salary2['Eng_score'] = salary2['English'].map(score_map)
salary2 = salary2.sort_values('Eng_score')

sns.relplot(x='Salary', y='Age', col='English', hue='Education',
            data=salary2, col_wrap=3, height=3)
plt.show()

---
## 3. 样式 & 环境设置

### 3.1 `set_theme()` / `axes_style()` — 五种绘图风格

In [ ]:
styles = ['darkgrid', 'whitegrid', 'dark', 'white', 'ticks']
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, s in zip(axes, styles):
    with sns.axes_style(s):  # 临时切换，不影响全局
        ax.plot([1, 2, 3, 4], [1, 4, 9, 16], 'r-o')
        ax.set_title(f"'{s}'")
plt.suptitle('五种风格 (with axes_style 临时)', y=1.05)
plt.tight_layout(); plt.show()

In [ ]:
# 全局 vs 局部
sns.set_theme(style='whitegrid')  # 全局设置

plt.figure(figsize=(8, 3))
with sns.axes_style('darkgrid'):  # 只在 with 内生效
    plt.plot([1, 2, 3], [1, 4, 9], 'r-o')
    plt.title('with 内: darkgrid')
plt.figure(figsize=(8, 3))
plt.plot([1, 2, 3], [1, 4, 9], 'b-s')
plt.title('with 外: 恢复全局 whitegrid')
plt.show()
# sns.reset_defaults()  # 恢复 matplotlib 默认

### 3.2 `set_context()` — 缩放级别 & `font_scale`

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, ctx in zip(axes, ['paper', 'notebook', 'talk', 'poster']):
    with sns.plotting_context(ctx):
        ax.plot([1, 2, 3], [1, 4, 9], 'r-o')
        ax.set_title(f"'{ctx}'")
plt.suptitle('四种 context', y=1.05)
plt.tight_layout(); plt.show()

# 快捷字体缩放（不需要切换 context）
# sns.set(font_scale=1.5)  # 全局放大 1.5 倍

### 3.3 `despine()` — 去边框

In [ ]:
scores_df = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\scores.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
sns.boxplot(data=scores_df, ax=axes[0])
axes[0].set_title('默认 四边框')

sns.boxplot(data=scores_df, ax=axes[1])
sns.despine(left=True, right=True, top=True)
axes[1].set_title('despine(上左右) 只留底部')
plt.show()
# sns.despine(left=False, right=False, top=False)  # 恢复

### 3.4 `sns.move_legend()` — 移动图例位置

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv', index_col=0)
ax = sns.scatterplot(x='Salary', y='Age', hue='Education', data=salary)
# 把图例移到图表外面右上角
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='学历')
plt.title('move_legend 移动图例到外部')
plt.tight_layout(); plt.show()

---
## 4. 配色方案

### 4.1 分类调色板 `color_palette()` — 离散数据用

In [ ]:
names = ['deep', 'muted', 'bright', 'pastel', 'dark', 'colorblind']
fig, axes = plt.subplots(2, 3, figsize=(14, 5))
for ax, name in zip(axes.flat, names):
    sns.palplot(sns.color_palette(name), ax=ax)
    ax.set_title(f"'{name}'")
plt.suptitle('分类调色板 — palette= 参数直接使用', y=1.05)
plt.tight_layout(); plt.show()

### 4.2 连续调色板 `light_palette()` / `dark_palette()` — 数值渐变

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
sns.palplot(sns.light_palette('orange', n_colors=10), ax=axes[0])
axes[0].set_title("light_palette('orange', 10)")
sns.palplot(sns.dark_palette('teal', n_colors=10, reverse=True), ax=axes[1])
axes[1].set_title("dark_palette('teal', reverse=True)")
sns.palplot(sns.cubehelix_palette(10, start=0.5, rot=-0.5), ax=axes[2])
axes[2].set_title('cubehelix_palette(10)')
plt.tight_layout(); plt.show()

### 4.3 离散调色板 `diverging_palette()` / `color_palette('coolwarm')` — 双极对比

适合有正负值、高低对比的数据

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
sns.palplot(sns.color_palette('coolwarm', 10), ax=axes[0])
axes[0].set_title("color_palette('coolwarm', 10)")
sns.palplot(sns.diverging_palette(240, 10, n=10), ax=axes[1])
axes[1].set_title('diverging_palette(240, 10, n=10)')
sns.palplot(sns.color_palette('Spectral', 10), ax=axes[2])
axes[2].set_title("color_palette('Spectral', 10)")
plt.tight_layout(); plt.show()

### 4.4 配色实战：as_cmap=True 生成 colormap（热力图用）

In [ ]:
# as_cmap=True 生成 matplotlib colormap 对象
cmap1 = sns.light_palette('orange', as_cmap=True)      # 连渐变色
cmap2 = sns.color_palette('coolwarm', as_cmap=True)    # 离散 as cmap

# sns.heatmap(data, cmap=cmap1)  # 见热力图章节

---
## 5. 数据重塑 — melt & pivot

**Seaborn 要求长格式数据**（数值列 + 类别列）。
`melt` 宽→长 | `pivot` 长→矩阵（热力图用）| `pd.Categorical` 固定顺序。

### 5.1 手动拼接（小数据快速转换）

In [ ]:
scores = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\scores.csv')

ga = scores.iloc[:, 0].tolist(); gb = scores.iloc[:, 1].tolist()
gc = scores.iloc[:, 2].tolist(); gd = scores.iloc[:, 3].tolist()

long_df = pd.DataFrame({
    'Group': ['A']*len(ga) + ['B']*len(gb) + ['C']*len(gc) + ['D']*len(gd),
    'IQ': ga + gb + gc + gd
})

sns.boxplot(x='Group', y='IQ', data=long_df, palette='muted')
sns.despine(left=True, right=True, top=True)
plt.title('手动拼接 宽→长')
plt.show()

### 5.2 `pd.melt()` — 一键宽转长（推荐）

In [ ]:
movie = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\练习\movie_scores.csv')

# id_vars=保留的列, value_vars=要融化的列, var_name=新类别列名, value_name=新数值列名
movie_long = movie.melt(
    id_vars='MovieTitle',
    value_vars=['Tomatometer', 'AudienceScore'],
    var_name='ScoreType', value_name='Score'
)
movie_long.head(6)

In [ ]:
# melt 后的数据可直接画对比图
plt.figure(figsize=(10, 5), dpi=120)
sns.barplot(x='MovieTitle', y='Score', hue='ScoreType', data=movie_long, palette='muted')
plt.xticks(rotation=15)
plt.title('melt 后一键对比')
plt.show()

### 5.3 `pd.pivot()` — 长转矩阵（热力图必备）

In [ ]:
titanic = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\Seaborn data\titanic.csv')
agg = titanic.groupby(['Class', 'Sex']).agg({'Freq': 'sum'}).reset_index()
pivoted = agg.pivot(index='Class', columns='Sex', values='Freq')
pivoted

### 5.4 `pd.Categorical` — 固定分类顺序

In [ ]:
# 创建有序分类（字母序不再管用）
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df = pd.DataFrame({'month': ['Mar', 'Jan', 'Apr', 'Feb'], 'val': [10, 20, 30, 40]})
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)
df = df.sort_values('month')
df  # 现在 Jan -> Feb -> Mar -> Apr

---
## 6. 热力图 Heatmap & Clustermap

| 函数 | 特点 |
|------|------|
| `heatmap()` | 矩阵热力图，Axes-level |
| `clustermap()` | 带层次聚类的热力图，Figure-level |

### 6.1 heatmap — 基础

In [ ]:
titanic = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\Seaborn data\titanic.csv')

agg = titanic.groupby(['Class', 'Survived']).agg({'Freq': 'sum'}).reset_index()
heat_data = agg.pivot(index='Class', columns='Survived', values='Freq')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

sns.heatmap(heat_data, annot=True, fmt='d', linewidths=2,
            cmap=sns.light_palette('orange', as_cmap=True), ax=axes[0])
axes[0].set_title('各舱位 生还/遇难 人数')

# 进阶：生还率
agg2 = titanic.groupby(['Class', 'Sex', 'Survived']).agg({'Freq': 'sum'}).reset_index()
y = agg2[agg2['Survived'] == 'Yes'][['Class', 'Sex', 'Freq']]
n = agg2[agg2['Survived'] == 'No'][['Class', 'Sex', 'Freq']]
m = y.merge(n, on=['Class', 'Sex'], suffixes=('_y', '_n'))
m['rate'] = m['Freq_y'] / (m['Freq_y'] + m['Freq_n'])
rate_data = m.pivot(index='Class', columns='Sex', values='rate')

sns.heatmap(rate_data, annot=True, fmt='.1%', linewidths=2,
            cmap=sns.light_palette('green', as_cmap=True),
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('各舱位x性别 生还率')

plt.tight_layout(); plt.show()

### 6.2 航班旅客热力图（含月份 Categorical 排序）

In [ ]:
flights = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\Seaborn data\flight_details.csv')

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
flights['Months'] = pd.Categorical(flights['Months'], categories=month_order, ordered=True)
fp = flights.pivot(index='Months', columns='Years', values='Passengers')

plt.figure(figsize=(14, 6), dpi=150)
sns.heatmap(fp, annot=True, fmt='d', linewidths=0.5,
            cmap='YlOrRd', cbar_kws={'label': '旅客人数'})
plt.title('航空旅客人数热力图 (2001-2012)', fontsize=16, fontweight='bold')
plt.xlabel('年份'); plt.ylabel('月份')
plt.tight_layout(); plt.show()

### 6.3 clustermap — 层次聚类热力图

自动对行和列做层次聚类，发现数据中的分组结构。

In [ ]:
# 用航班数据演示 clustermap
flights = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\Seaborn data\flight_details.csv')
flights['Months'] = pd.Categorical(flights['Months'], categories=month_order, ordered=True)
fp = flights.pivot(index='Months', columns='Years', values='Passengers')

# clustermap: 自动层次聚类 + 树状图
sns.clustermap(fp, annot=True, fmt='d', cmap='YlOrRd',
               linewidths=0.5, figsize=(12, 10),
               col_cluster=False,  # 不对列（年份）聚类
               standard_scale=1)    # 按行标准化 (0=按列, 1=按行)
plt.suptitle('clustermap — 层次聚类热力图 (行标准化)', y=1.02)
plt.show()

---
## 7. 柱状图 Barplot

自动计算均值 + 误差线（置信区间）。`hue` 分组，`dodge` 是否错开。

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(x='Education', y='Salary', hue='District', data=salary, ax=axes[0], palette='muted')
axes[0].set_title('教育 vs 收入 (hue=地区)'); axes[0].tick_params(axis='x', rotation=30)

sns.barplot(x='District', y='Salary', hue='Education', data=salary, ax=axes[1], palette='Set2')
axes[1].set_title('地区 vs 收入 (hue=教育) 换个角度')

plt.tight_layout(); plt.show()

---
## 8. 箱线图 & 提琴图 & 箱线增强图

| 图表 | 函数 | 特点 |
|------|------|------|
| 箱线图 | `boxplot()` | Q1/Q2/Q3/IQR/离群值 |
| 提琴图 | `violinplot()` | 箱线+核密度，看分布形状 |
| 增强箱线 | `boxenplot()` | 更多分位数，适合大数据 |

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(x='Education', y='Salary', hue='Gender', data=salary, ax=axes[0], palette='Set2')
axes[0].set_title('boxplot 箱线图'); axes[0].tick_params(axis='x', rotation=30)

sns.violinplot(x='Education', y='Salary', hue='Gender', data=salary, ax=axes[1], palette='muted')
axes[1].set_title('violinplot 提琴图'); axes[1].tick_params(axis='x', rotation=30)

sns.violinplot(x='Education', y='Salary', hue='Gender', data=salary, ax=axes[2],
               split=True, palette='pastel')
axes[2].set_title('violinplot split=True 对半'); axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

In [ ]:
# boxenplot — 增强箱线图，更多分位数
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxenplot(x='Education', y='Salary', hue='Gender', data=salary, palette='Set2')
ax.set_title('boxenplot — 增强箱线图'); ax.tick_params(axis='x', rotation=30)
plt.show()

---
## 9. 分布图 — histplot / kdeplot / ecdfplot / rugplot

| 函数 | 作用 |
|------|------|
| `histplot()` | 直方图 |
| `kdeplot()` | 核密度估计曲线 |
| `ecdfplot()` | 经验累积分布函数 |
| `rugplot()` | 地毯图（轴上的小竖线标记） |
| `displot()` | Figure-level 统一入口 |

In [ ]:
np.random.seed(42)
x = np.random.normal(0, 1, 2000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# histplot
sns.histplot(x, bins=40, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(x=0, color='red', linestyle='--', label='mean=0')
axes[0].set_title('histplot + KDE'); axes[0].legend()

# kdeplot
x2 = np.random.normal(1.5, 0.8, 1500)
sns.kdeplot(x=x, fill=True, alpha=0.3, label='N(0,1)', ax=axes[1])
sns.kdeplot(x=x2, fill=True, alpha=0.3, label='N(1.5,0.8)', ax=axes[1])
axes[1].set_title('kdeplot 两组对比'); axes[1].legend()

# kdeplot + rugplot 组合（地毯图标记每个数据点）
sample = np.random.normal(0, 1, 100)
sns.kdeplot(x=sample, fill=True, alpha=0.3, ax=axes[2])
sns.rugplot(x=sample, height=0.05, alpha=0.5, ax=axes[2])
axes[2].set_title('kdeplot + rugplot 地毯图')

plt.tight_layout(); plt.show()

In [ ]:
# ecdfplot — 累积分布
fig, ax = plt.subplots(figsize=(8, 4))
sns.ecdfplot(x, color='steelblue', label='N(0,1)')
sns.ecdfplot(x2, color='coral', label='N(1.5,0.8)')
ax.set_title('ecdfplot — 经验累积分布函数'); ax.legend()
plt.show()

In [ ]:
# displot — Figure-level，自动按 hue 分面
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')
sns.displot(salary, x='Salary', hue='Education', kind='kde', fill=True,
            palette='Set2', height=5, aspect=1.5)
plt.title('displot — 按教育水平看工资分布'); plt.show()

---
## 10. 联合分布 Jointplot

同时展示两个变量的散点 + 各自边缘分布。`kind=` → `'scatter'` / `'kde'` / `'hist'` / `'reg'` / `'hex'`

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

# scatter
sns.jointplot(x='Salary', y='Age', data=salary[:200], kind='scatter', height=6)
plt.suptitle('jointplot scatter', y=1.02); plt.show()

# kde
sns.jointplot(x='Salary', y='Age', data=salary[:200], kind='kde', fill=True, height=6)
plt.suptitle('jointplot kde', y=1.02); plt.show()

# reg
sns.jointplot(x='Salary', y='Age', data=salary[:200], kind='reg', height=6)
plt.suptitle('jointplot reg', y=1.02); plt.show()

# hex — 六边形分箱（大数据好用）
sns.jointplot(x='Salary', y='Age', data=salary, kind='hex', height=6,
              marginal_kind='hist')
plt.suptitle('jointplot hex', y=1.02); plt.show()

---
## 11. 配对图 Pairplot

所有数值列两两配对画散点 + 对角画分布。`hue` 按类别上色。`corner=True` 只画下半三角。

In [ ]:
details = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\Seaborn data\basic_details.csv')

# 全变量配对
sns.pairplot(details, hue='Groups', palette='Set2', diag_kind='kde', height=2.5,
             plot_kws={'alpha': 0.6})
plt.suptitle('pairplot 全变量矩阵', y=1.02); plt.show()

In [ ]:
# 指定列 + 下半三角
sns.pairplot(details, vars=['Weight', 'Height', 'Age'], hue='Groups',
             palette='muted', corner=True, height=3)
plt.suptitle('pairplot 指定列 + corner', y=1.02); plt.show()

---
## 12. FacetGrid — 自由分面子图

比 relplot 的 col/row 更灵活：`.map()` 可以映射**任何**绘图函数（plt.scatter, sns.histplot, plt.plot 等）。

**三步走**：创建网格 → `.map(函数, 参数...)` → `.add_legend()` / `.set_titles()`

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

g = sns.FacetGrid(salary, row='Gender', col='District', hue='Education',
                  height=3, aspect=1.2, sharex=False, sharey=False)
g.map(plt.scatter, 'Salary', 'Age', alpha=0.6, s=30)
g.add_legend()
g.set_titles('{row_name} — {col_name}')
g.set_axis_labels('Salary', 'Age')
plt.show()

In [ ]:
# map 不同图表类型
g = sns.FacetGrid(salary, col='Education', col_wrap=3, height=3.5, sharex=True)
g.map(sns.histplot, 'Salary', bins=15, alpha=0.7, color='steelblue')
g.set_titles('{col_name}')
plt.show()

---
## 13. 回归图 — regplot / lmplot

| 函数 | 级别 | 分面 |
|------|------|------|
| `regplot()` | Axes-level | 不支持 |
| `lmplot()` | Figure-level | hue/col/row 分面 |

**关键参数**：`order=` 多项式阶数 / `ci=` 置信区间 / `logistic=True` 逻辑回归

In [ ]:
anage = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\多子图\anage_data(1).csv')
mammals = anage[(anage['Class'] == 'Mammalia') &
                (anage['Body mass (g)'] < 200000) &
                (anage['Maximum longevity (yrs)'].notna())].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=120)

# regplot 基础
sns.regplot(x='Body mass (g)', y='Maximum longevity (yrs)', data=mammals, ax=axes[0],
            scatter_kws={'alpha': 0.3, 's': 15}, line_kws={'color': 'red', 'linewidth': 2})
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_title('regplot — 体重 vs 寿命 (双对数)')

# regplot 多项式回归 + 去掉置信区间
sns.regplot(x='Body mass (g)', y='Maximum longevity (yrs)', data=mammals, ax=axes[1],
            order=2, ci=None, scatter_kws={'alpha': 0.3, 's': 15},
            line_kws={'color': 'green', 'linewidth': 2, 'linestyle': '--'})
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_title('regplot — 二次多项式 order=2 (不显示 CI)')

plt.tight_layout(); plt.show()

In [ ]:
# lmplot — 带分面的回归图
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')
sns.lmplot(x='Age', y='Salary', hue='Gender', col='Education',
           data=salary, height=3.5, col_wrap=3,
           scatter_kws={'alpha': 0.3}, palette='Set1')
plt.show()

---
## 14. Catplot — 分类图统一入口

`sns.catplot(kind='...')` 统一调用。`kind=` 可选：`'strip'` / `'swarm'` / `'box'` / `'violin'` / `'boxen'` / `'point'` / `'bar'` / `'count'`。自带 col/row 分面。

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
kinds = [
    ('strip', '散点抖动'), ('swarm', '蜂群不重叠'), ('box', '箱线图'),
    ('violin', '提琴图'), ('boxen', '增强箱线'), ('bar', '柱状图')
]
for ax, (kind, title) in zip(axes.flat, kinds):
    sns.catplot(x='Education', y='Salary', kind=kind, data=salary, ax=ax)
    ax.set_title(f'kind={kind} — {title}'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# catplot 分面
sns.catplot(x='Education', y='Salary', kind='box', hue='Gender',
            col='District', data=salary, height=4, col_wrap=2, palette='Set2')
plt.show()

### countplot — 计数柱状图（不需要 y 值，自动统计频次）

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Education', data=salary, palette='muted', ax=axes[0])
axes[0].set_title('countplot — 每个教育水平的数量')
sns.countplot(x='Education', hue='Gender', data=salary, palette='Set2', ax=axes[1])
axes[1].set_title('countplot + hue')
plt.tight_layout(); plt.show()

### pointplot — 点估计 + 误差线（看趋势/对比）

In [ ]:
salary = pd.read_csv(r'd:\VScode\jupyter_program\数据可视化\Seaborn\salary.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 基础
sns.pointplot(x='Education', y='Salary', data=salary, ax=axes[0],
              color='steelblue', markers='D', linestyles='-')
axes[0].set_title('pointplot — 均值 + 置信区间'); axes[0].tick_params(axis='x', rotation=30)

# hue + dodge 分组
sns.pointplot(x='Education', y='Salary', hue='Gender', data=salary, ax=axes[1],
              dodge=True, palette='Set1', markers=['o', 's'])
axes[1].set_title('pointplot + hue 分组对比'); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

---
## 15. 树形图 Squarify（需要 `pip install squarify`）

面积大小表示数值比例，配合 Seaborn 调色板。

In [ ]:
import squarify

plt.figure(figsize=(10, 7), dpi=120)
sizes = [17, 12, 20, 19, 24, 8]
labels = ['Washing\n17%', 'Leak\n12%', 'Faucet\n20%',
          'Shower\n19%', 'Toilet\n24%', 'Other\n8%']
squarify.plot(sizes=sizes, label=labels, alpha=0.85,
              color=sns.color_palette('Set2', len(sizes)),
              edgecolor='white', linewidth=3,
              text_kwargs={'fontsize': 13, 'fontweight': 'bold'})
plt.axis('off')
plt.title('家庭用水 — 树形图', fontsize=15, fontweight='bold')
plt.show()

---
## 16. 综合实战 — YouTube 频道分析

一次性展示 FacetGrid + 多图表类型组合。

In [ ]:
# YouTube Top30 数据
channels = ['PewDiePie', 'T-Series', '5-Minute Crafts', 'Canal KondZilla', 'Justin Bieber',
            'SET India', 'WWE', 'Dude Perfect', 'HolaSoyGerman.', 'Ed Sheeran',
            'EminemMusic', 'Badabun', 'Cocomelon', 'whinderssonnunes', 'JustinBieberVEVO',
            'elrubiusOMG', 'JuegaGerman', 'Taylor Swift', 'Katy Perry', 'Fernanfloo',
            'Ariana Grande', 'Rihanna', 'TheEllenShow', 'Zee Music', 'Felipe Neto',
            'One Direction', 'YouTube Spotlight', 'TaylorSwiftVEVO', 'EminemVEVO', 'KatyPerryVEVO']
subs = [83.1, 82.9, 48, 46.1, 43.1, 40.7, 39.2, 38.7, 36.9, 36.8,
        35.8, 35, 34.8, 34.2, 33.9, 33.5, 32.7, 32.2, 32.1, 31.8,
        31.6, 31.2, 30.6, 30.3, 30.2, 28.7, 28, 27.8, 26.4, 26.4]
views = [20329, 61057, 12061, 22878, 601, 28573, 29886, 7168, 3790, 15926,
         637, 19388, 9737, 2801, 18375, 7313, 8730, 233, 357, 706,
         6755, 59, 14847, 14614, 5938, 330, 1833, 16161, 12548, 16890]

df_yt = pd.DataFrame({'Channel': channels, 'Subscribers_M': subs, 'Views_M': views})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1) 订阅量 Top10 柱状图
top10 = df_yt.nlargest(10, 'Subscribers_M')
sns.barplot(x='Subscribers_M', y='Channel', data=top10,
            palette=sns.light_palette('steelblue', n_colors=10, reverse=True), ax=axes[0])
axes[0].set_title('订阅量 Top10'); axes[0].set_xlabel('M')

# 2) 订阅 vs 观看 散点图 + 回归线
sns.regplot(x='Subscribers_M', y='Views_M', data=df_yt, ax=axes[1],
            scatter_kws={'alpha': 0.6, 's': 60}, line_kws={'color': 'red'})
axes[1].set_title('订阅 vs 观看量 (regplot)'); axes[1].set_xlabel('订阅量 (M)'); axes[1].set_ylabel('观看量 (M)')

# 3) 观看量分布
sns.histplot(df_yt['Views_M'], bins=15, kde=True, ax=axes[2], color='coral')
axes[2].axvline(df_yt['Views_M'].median(), color='blue', linestyle='--', label=f"中位数={df_yt['Views_M'].median():.0f}M")
axes[2].set_title('观看量分布'); axes[2].legend()

plt.tight_layout(); plt.show()

---
## 速查表 Quick Reference

### Figure-level vs Axes-level
| Figure-level | Axes-level |
|---|---|
| `relplot()` | `scatterplot()`, `lineplot()` |
| `displot()` | `histplot()`, `kdeplot()`, `ecdfplot()`, `rugplot()` |
| `catplot()` | `boxplot()`, `violinplot()`, `barplot()`, `stripplot()`, `swarmplot()`, `pointplot()`, `boxenplot()`, `countplot()` |
| `lmplot()` | `regplot()` |
| `jointplot()` | -- |
| `pairplot()` | -- |
| `clustermap()` | `heatmap()` |

### 四大分组参数
| 参数 | 作用 | 适用函数 |
|------|------|------|
| `hue` | 颜色分组 | relplot/catplot/lmplot/displot/jointplot/pairplot |
| `style` | 点形/线形 | relplot |
| `size` | 大小 | relplot |
| `col` / `row` | 分面 | relplot/catplot/lmplot/displot |
| `hue_order` / `col_order` | 分组顺序 | 以上所有 |

### 样式
| 代码 | 说明 |
|------|------|
| `sns.set_theme(style='whitegrid')` | 全局主题 |
| `with sns.axes_style('darkgrid'):` | 临时样式 |
| `sns.set_context('talk')` | 缩放(paper/notebook/talk/poster) |
| `sns.set(font_scale=1.5)` | 字体缩放 |
| `sns.despine(left, top, right)` | 去边框 |
| `sns.move_legend(ax, 'upper left', bbox_to_anchor=(1,1))` | 移动图例 |
| `sns.reset_defaults()` | 恢复默认 |

### 配色
| 代码 | 类型 |
|------|------|
| `sns.color_palette('muted')` | 分类 |
| `sns.light_palette('orange', as_cmap=True)` | 连续/热力图 cmap |
| `sns.diverging_palette(h1, h2, n=)` | 双极对比 |
| `sns.color_palette('coolwarm', n)` | 离散色 |
| `sns.cubehelix_palette(n, start=, rot=)` | 线性亮度渐变 |

### 数据重塑
| 代码 | 说明 |
|------|------|
| `df.melt(id_vars, value_vars, var_name, value_name)` | 宽→长 |
| `df.pivot(index, columns, values)` | 长→矩阵(热力图) |
| `pd.Categorical(col, categories=order, ordered=True)` | 固定顺序 |

### 热力图
| 参数 | 说明 |
|------|------|
| `annot=True, fmt='d'/'%.2f'/'%.1%'` | 显示数值 |
| `linewidths` / `linecolor` | 格子间距/颜色 |
| `cmap=` | 颜色映射 (需 as_cmap=True) |
| `vmin/vmax` | 颜色值范围 |
| `cbar_kws={'label': ''}` | 颜色条设置 |
| `standard_scale=0/1` | clustermap 行列标准化 |

### 回归图
| 参数 | 说明 |
|------|------|
| `order=1` | 多项式阶数 |
| `ci=95` | 置信区间(None=不显示) |
| `logistic=True` | 逻辑回归 |
| `scatter_kws={}` / `line_kws={}` | 样式 |

### FacetGrid
| 代码 | 说明 |
|------|------|
| `g = sns.FacetGrid(data, row, col, hue)` | 创建网格 |
| `g.map(func, *args)` | 映射绘图函数 |
| `g.add_legend()` | 添加图例 |
| `g.set_titles()` / `g.set_axis_labels()` | 标题/标签 |

---

> **复习建议**：
> 1. 记住 Figure-level (自动图) vs Axes-level (可嵌入子图) 的区别
> 2. **hue / style / size / col / row** 五个参数可以组合出无穷变化
> 3. 数据预处理：**melt(宽→长)** 和 **pivot(长→矩阵)** 是必备前处理
> 4. 配色三步走：分类用 muted/deep → 连续用 light_palette → 双极用 diverging_palette
> 5. [Seaborn 官方 Gallery](https://seaborn.pydata.org/examples/index.html) 是最好的进阶教材